# 🪟 Window Functions (Funções de Janela)

Window Functions realizam cálculos em um conjunto de linhas relacionadas à linha atual,
**sem** agrupar os resultados como o `GROUP BY`.

> 💡 Diferente do `GROUP BY`, as Window Functions **mantêm todas as linhas** do resultado. 
---

In [2]:
# Importando as bibliotecas necessárias

import pandas as pd
import sqlite3
from sqlite3 import Error

In [3]:
def create_connection():
    conn = None
    try:
           # Cria (ou conecta) ao banco de dados local
        conn = sqlite3.connect("vendas.db")
        print(f"Conexão bem sucedida com Sqlite {sqlite3.sqlite_version}")
    except Error as e:
        print(f'Erro {e} na conexão com Sqlite')
    return conn

In [4]:
def create_table(conn):
    try:
        # Tabela de vendas com vendedor, produto, quantidade e data
        query = '''
            CREATE TABLE vendas (
                ID INTEGER PRIMARY KEY,
                Vendedor TEXT NOT NULL,
                Produto TEXT NOT NULL,
                Quantidade INTEGER,
                Data_da_Venda DATE NOT NULL
            );
        '''
        conn.execute(query)
        print('Tabela criada com sucesso')
    except Error as e:
        print(f'Erro {e} na criação da tabela')

In [5]:
def insert_data(conn):
    try:
        # Inserindo 10 registros de vendas para 3 vendedores: Ana, Beto e Carlos
        query = '''
            INSERT INTO vendas(ID, Vendedor, Produto, Quantidade, Data_da_Venda)
            VALUES
            (1, 'Ana', 'Livro', 10, '2026-01-01'),
            (2, 'Beto', 'Lápis', 30, '2026-01-05'),
            (3, 'Carlos', 'Caderno', 15, '2026-01-08'),
            (4, 'Ana', 'Caderno', 20, '2026-01-09'),
            (5, 'Beto', 'Caneta', 50, '2026-01-12'),
            (6, 'Carlos', 'Livro', 25, '2026-01-15'),
            (7, 'Ana', 'Caneta', 30, '2026-01-17'),
            (8, 'Beto', 'Caderno', 40, '2026-01-19'),
            (9, 'Carlos', 'Lápis', 35, '2026-01-22'),
            (10, 'Ana', 'Livro', 45, '2026-01-25');
        '''
        conn.execute(query)
        conn.commit()
        print('Dados inseridos com sucesso')
    except Error as e:
        print(f'Erro {e} na inserção de dados')

In [6]:
conn = create_connection()

with conn:
    create_table(conn)
    insert_data(conn)

Conexão bem sucedida com Sqlite 3.50.4
Erro table vendas already exists na criação da tabela
Erro UNIQUE constraint failed: vendas.ID na inserção de dados


In [7]:
def exectute_query(conn, query):
    try:
          # Executa uma query SQL e retorna o resultado em um DataFrame pandas
        df = pd.read_sql(query, conn)
        display(df)
    except Error as e:
        print(f'Erro {e} na execução da query')

## 🔍 Praticando Window Functions

Abaixo, apliquei as funções de janela na tabela `vendas` para analisar
o desempenho dos vendedores ao longo do tempo.

In [8]:
# Venda total usando OVER sem PARTITION (soma de toda a tabela repetida em cada linha)

with conn:
    print("Query 1: Função OVER")
    query1 = '''
        SELECT Vendedor, Produto, Quantidade, Data_da_Venda,
         -- SUM com OVER vazio = soma de TODA a tabela repetida em cada linha
        SUM(Quantidade) OVER () AS Quantidade_Total
        -- OVER sem PARTITION = Janela que abrange toda a tabela
        FROM vendas;
    '''
    exectute_query(conn, query1)

Query 1: Função OVER


,Vendedor,Produto,Quantidade,Data_da_Venda,Quantidade_Total
0,Ana,Livro,10,2026-01-01,300
1,Beto,Lápis,30,2026-01-05,300
2,Carlos,Caderno,15,2026-01-08,300
3,Ana,Caderno,20,2026-01-09,300
4,Beto,Caneta,50,2026-01-12,300
5,Carlos,Livro,25,2026-01-15,300
6,Ana,Caneta,30,2026-01-17,300
7,Beto,Caderno,40,2026-01-19,300
8,Carlos,Lápis,35,2026-01-22,300
9,Ana,Livro,45,2026-01-25,300


In [9]:
# Média total de quantidade vendida usando OVER sem PARTITION

with conn:
    print("Query 2: Função OVER")
    query2 = '''
        SELECT Vendedor, Produto, Quantidade, Data_da_Venda,
        AVG(Quantidade) OVER () AS Média_Total
        FROM vendas;
    '''
    exectute_query(conn, query2)

Query 2: Função OVER


,Vendedor,Produto,Quantidade,Data_da_Venda,Média_Total
0,Ana,Livro,10,2026-01-01,30.0
1,Beto,Lápis,30,2026-01-05,30.0
2,Carlos,Caderno,15,2026-01-08,30.0
3,Ana,Caderno,20,2026-01-09,30.0
4,Beto,Caneta,50,2026-01-12,30.0
5,Carlos,Livro,25,2026-01-15,30.0
6,Ana,Caneta,30,2026-01-17,30.0
7,Beto,Caderno,40,2026-01-19,30.0
8,Carlos,Lápis,35,2026-01-22,30.0
9,Ana,Livro,45,2026-01-25,30.0


In [10]:
# Máximo total de quantidade vendida usando OVER sem PARTITION

with conn:
    print("Query 3: Função OVER")
    query3 = '''
        SELECT Vendedor, Produto, Quantidade, Data_da_Venda,
        MAX(Quantidade) OVER () AS Maximo_Total
        FROM vendas;
    '''
    exectute_query(conn, query3)

Query 3: Função OVER


,Vendedor,Produto,Quantidade,Data_da_Venda,Maximo_Total
0,Ana,Livro,10,2026-01-01,50
1,Beto,Lápis,30,2026-01-05,50
2,Carlos,Caderno,15,2026-01-08,50
3,Ana,Caderno,20,2026-01-09,50
4,Beto,Caneta,50,2026-01-12,50
5,Carlos,Livro,25,2026-01-15,50
6,Ana,Caneta,30,2026-01-17,50
7,Beto,Caderno,40,2026-01-19,50
8,Carlos,Lápis,35,2026-01-22,50
9,Ana,Livro,45,2026-01-25,50


In [11]:
# Função PARTITION BY para calcular a soma de quantidade por vendedor

with conn:
    print("Query 4: Função PARTITION BY")
    query4 = '''
        SELECT Vendedor, Produto, Quantidade, Data_da_Venda,
             -- PARTITION BY Vendedor = divide a janela por vendedor
        -- cada vendedor tem sua própria soma, repetida em suas linhas
        SUM(Quantidade) OVER (PARTITION BY Vendedor) AS Quantidade_Total_Por_Vendedor
        FROM vendas;
    '''
    exectute_query(conn, query4)

Query 4: Função PARTITION BY


,Vendedor,Produto,Quantidade,Data_da_Venda,Quantidade_Total_Por_Vendedor
0,Ana,Livro,10,2026-01-01,105
1,Ana,Caderno,20,2026-01-09,105
2,Ana,Caneta,30,2026-01-17,105
3,Ana,Livro,45,2026-01-25,105
4,Beto,Lápis,30,2026-01-05,120
5,Beto,Caneta,50,2026-01-12,120
6,Beto,Caderno,40,2026-01-19,120
7,Carlos,Caderno,15,2026-01-08,75
8,Carlos,Livro,25,2026-01-15,75
9,Carlos,Lápis,35,2026-01-22,75


In [12]:
# Função PARTITION BY para calcular a média de quantidade por vendedor, ordenada pela média

with conn:
    print("Query 5: Função PARTITION BY")
    query5 = '''
        SELECT Vendedor, Produto, Quantidade, Data_da_Venda,
        AVG(Quantidade) OVER (PARTITION BY Vendedor) AS Media_Por_Vendedor
        FROM vendas
        ORDER BY Media_Por_Vendedor DESC;
    '''
    exectute_query(conn, query5)

Query 5: Função PARTITION BY


,Vendedor,Produto,Quantidade,Data_da_Venda,Media_Por_Vendedor
0,Beto,Lápis,30,2026-01-05,40.00
1,Beto,Caneta,50,2026-01-12,40.00
2,Beto,Caderno,40,2026-01-19,40.00
3,Ana,Livro,10,2026-01-01,26.25
4,Ana,Caderno,20,2026-01-09,26.25
5,Ana,Caneta,30,2026-01-17,26.25
6,Ana,Livro,45,2026-01-25,26.25
7,Carlos,Caderno,15,2026-01-08,25.00
8,Carlos,Livro,25,2026-01-15,25.00
9,Carlos,Lápis,35,2026-01-22,25.00


In [13]:
# Função RANK para classificar os vendedores por quantidade vendida, permitindo empates

with conn:
    print("Query 6: Função RANK (Aceita empates)")
    query6 = '''
        SELECT Vendedor, Produto, Quantidade, Data_da_Venda,
        RANK() OVER (ORDER BY Quantidade DESC) AS Rank -- Continua sendo em janela (mesmo que ordenado)
        FROM vendas;
    '''
    
    exectute_query(conn, query6)

Query 6: Função RANK (Aceita empates)


,Vendedor,Produto,Quantidade,Data_da_Venda,Rank
0,Beto,Caneta,50,2026-01-12,1
1,Ana,Livro,45,2026-01-25,2
2,Beto,Caderno,40,2026-01-19,3
3,Carlos,Lápis,35,2026-01-22,4
4,Beto,Lápis,30,2026-01-05,5
5,Ana,Caneta,30,2026-01-17,5
6,Carlos,Livro,25,2026-01-15,7
7,Ana,Caderno,20,2026-01-09,8
8,Carlos,Caderno,15,2026-01-08,9
9,Ana,Livro,10,2026-01-01,10


In [14]:
# Função RANK para classificar os vendedores por quantidade vendida dentro de cada vendedor

with conn:
    print("Query 7: Função RANK por Vendedor + PARTITION")
    query7 = '''
        SELECT Vendedor, Produto, Quantidade, Data_da_Venda,
        -- PARTITION BY Vendedor = ranking reinicia para cada vendedor
        RANK() OVER (PARTITION BY Vendedor ORDER BY Quantidade DESC) AS Rank
        FROM vendas;
    '''
    exectute_query(conn, query7)

Query 7: Função RANK por Vendedor + PARTITION


,Vendedor,Produto,Quantidade,Data_da_Venda,Rank
0,Ana,Livro,45,2026-01-25,1
1,Ana,Caneta,30,2026-01-17,2
2,Ana,Caderno,20,2026-01-09,3
3,Ana,Livro,10,2026-01-01,4
4,Beto,Caneta,50,2026-01-12,1
5,Beto,Caderno,40,2026-01-19,2
6,Beto,Lápis,30,2026-01-05,3
7,Carlos,Lápis,35,2026-01-22,1
8,Carlos,Livro,25,2026-01-15,2
9,Carlos,Caderno,15,2026-01-08,3


In [15]:
# Função RANK para classificar os vendedores pelo total de quantidade vendida

with conn:
    print("Query 8: Função RANK")
    query8 = """
    SELECT Vendedor, SUM(Quantidade),
    RANK() OVER (ORDER BY SUM(Quantidade) DESC) AS Rank
    FROM vendas
    GROUP BY Vendedor;
    """
    exectute_query(conn, query8)

Query 8: Função RANK


,Vendedor,SUM(Quantidade),Rank
0,Beto,120,1
1,Ana,105,2
2,Carlos,75,3


In [16]:
# Função LAG para trazer a quantidade da venda anterior de cada vendedor

with conn:
    print("Query 10: Função LAG + PARTITION BY")
    query10 = """
    SELECT Vendedor, Produto, Data_da_Venda, Quantidade,
    LAG(Quantidade) OVER (PARTITION BY Vendedor ORDER BY Data_da_Venda) AS Quantidade_Anterior
    FROM vendas;
    """
    exectute_query(conn, query10)

Query 10: Função LAG + PARTITION BY


,Vendedor,Produto,Data_da_Venda,Quantidade,Quantidade_Anterior
0,Ana,Livro,2026-01-01,10,NaN
1,Ana,Caderno,2026-01-09,20,10.0
2,Ana,Caneta,2026-01-17,30,20.0
3,Ana,Livro,2026-01-25,45,30.0
4,Beto,Lápis,2026-01-05,30,NaN
5,Beto,Caneta,2026-01-12,50,30.0
6,Beto,Caderno,2026-01-19,40,50.0
7,Carlos,Caderno,2026-01-08,15,NaN
8,Carlos,Livro,2026-01-15,25,15.0
9,Carlos,Lápis,2026-01-22,35,25.0


In [17]:
conn.close()

---

## 📚 O que aprendi hoje

- ✅ Criar e conectar um banco de dados SQLite com Python
- ✅ Inserir dados com `INSERT INTO`
- ✅ Usar `OVER()` para calcular agregações sem agrupar linhas
- ✅ Usar `PARTITION BY` para dividir a janela por grupos
- ✅ Usar `RANK()` para classificar registros com e sem partição
- ✅ Usar `LAG()` para acessar valores de linhas anteriores

---